In [13]:
import pandas as pd
import os

In [14]:
original_date_format = "%a %b %d %H:%M:%S %Z %Y"
required_columns = ["Byline", "Body", "Headline", "Publish Date", "Publisher", "Paths"]
batch_size = 1000

In [15]:
def read_folder(folder_path):
    dfs = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            df = pd.read_csv(os.path.join(folder_path, filename))
            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df

In [80]:
def preprocess_df(df):
    if df.index.name is not None:
        df.reset_index(drop=False, inplace=True)

    df.rename(columns={"content": "Body", "title": "Headline", "pubDate": "Publish Date", "link": "Paths"}, inplace=True) 
     
    if 'Publisher' not in df.columns:
        df['Publisher'] = "Publisher"
        print("Publisher column not found. Filling with 'Publisher'")
    else:
        df['Publisher'] = df['Publisher'].fillna("Publisher")
        print("Publisher column not found. Filling with 'Publisher'")

    if "Byline" not in df.columns:
        df['Byline'] = "Author"
        print("Byline column not found. Filling with 'Author'")
    else:
        df['Byline'] = df['Byline'].fillna("Author")
        print("Byline column not found. Filling with 'Author'")
        
    if required_columns:
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

    # Ensure only the required columns are retained
    df = df[required_columns]
    print(f"df initial size: {df.shape}")
    
    def is_not_empty(value):
        return pd.notna(value) and str(value).strip() != ''

    for col in required_columns:
        df = df[df[col].apply(is_not_empty)]

    print(f"df size after removing empty rows: {df.shape}")

    df = df.dropna(subset=["Byline"])

    def parse_date(date_str):
        # Replace "EDT" with "-0400" and "EST" with "-0500"
        if "EDT" in date_str:
            date_str = date_str.replace("EDT", "-0400")
        elif "EST" in date_str:
            date_str = date_str.replace("EST", "-0500")
        
        try:
            # Parse the date string with the adjusted time zone
            return pd.to_datetime(date_str, errors='coerce')
        except ValueError:
            return None 
    # def parse_date(date_str):
    #     try:
    #         return pd.to_datetime(date_str, format=original_date_format, errors='coerce')
    #     except ValueError:
    #         return pd.to_datetime(date_str, infer_datetime_format=True, errors='coerce')

    # Apply date parsing function
    df['Publish Date'] = df['Publish Date'].apply(parse_date)

    df = df.dropna(subset=['Publish Date'])
    df = df.sort_values(by='Publish Date')

    print(f"df size after removing rows with invalid dates: {df.shape}")

    def format_date(date):
        if pd.notna(date):
            return date.strftime(original_date_format)
        return ''

    df['Publish Date'] = df['Publish Date'].apply(format_date)
    
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)
    print(f"df final size: {df.shape}")
    
    return df

In [83]:
def split_csv_into_batches(file_path):

    df = pd.read_csv(file_path)
    df = preprocess_df(df)
    
    total_rows = len(df)
    num_batches = (total_rows + batch_size - 1) // batch_size 
    # Split the DataFrame into batches and save each batch as a CSV
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, total_rows)
        batch_df = df.iloc[start_idx:end_idx]
        batch_file_name = f"./../data_set/batches/batch_{i+1}.csv"
        batch_df.to_csv(batch_file_name, index=False)
        print(f"Saved {batch_file_name} with {len(batch_df)} articles.")

    print("All batches have been saved successfully.")

In [55]:
# file_path = "./../data_set/Articles_Nov_2020_March_2023.csv"
# split_csv_into_batches(file_path)

In [58]:
folder_path = "./../data_set/data"
df1 = read_folder(folder_path)
df1 = preprocess_df(df1)
df1.to_csv("./../data_set/combined_data_1.csv", index=False)
df1

df initial size: (18696, 6)
df size after removing empty rows: (18239, 6)
df size after removing rows with invalid dates: (18128, 6)
df final size: (5985, 6)


,Byline,Body,Headline,Publish Date,Publisher,Paths
0,Adam Reilly,"If you’ve been around long enough, you still t...",LISTEN: Election 2020 — The Night(s) Before An...,Sun Nov 01 12:24:58 UTC-05:00 2020,Publisher,/politics/2020/11/01/listen-election-2020-the-...
1,Craig LeMoult,The state Department of Public Health released...,Household 'Clusters' Are A Problem In Massachu...,Sun Nov 01 12:51:24 UTC-05:00 2020,Publisher,/local-news/2020/11/01/household-clusters-are-...
2,Alana Wise,President Trump makes five stops in five diffe...,Trump And Biden Make 11th Hour Election Appeal...,Sun Nov 01 13:17:00 UTC-05:00 2020,Publisher,/politics/2020/11/01/trump-and-biden-make-11th...
3,"Arun Rath, Matt Baskin","<i>For the past year and a half, 20-year-old J...",20-Year-Old Massachusetts Man With Development...,Sun Nov 01 14:27:23 UTC-05:00 2020,Publisher,/local-news/2020/11/01/20-year-old-massachuset...
4,Matthew S. Schwartz,President Trump is celebrating a caravan of su...,Trump Speaks Fondly Of Supporters 'Protecting'...,Sun Nov 01 15:52:00 UTC-05:00 2020,Publisher,/politics/2020/11/02/trump-speaks-fondly-of-su...
...,...,...,...,...,...,...
5980,GBH News,"KYIV, Ukraine (AP) — Ukraine’s stunning incurs...",Ukraine gambled on an incursion deep into Russ...,Mon Aug 19 11:01:03 UTC-05:00 2024,Publisher,https://www.wgbh.org/news/international-news/2...
5981,GBH News,"This story, by Report for America corps member...",Vermont’s new motel room limits are primed to ...,Mon Aug 19 12:40:25 UTC-05:00 2024,Publisher,https://www.wgbh.org/news/local/2024-08-19/ver...
5982,GBH News,The Massachusetts emergency shelter system is ...,State's shelter system projected to run out of...,Mon Aug 19 21:26:53 UTC-05:00 2024,Publisher,https://www.wgbh.org/news/politics/2024-08-19/...
5983,GBH News,When Gov. Maura Healey announced last Friday t...,As state plans taking St. Elizabeth's Hospital...,Mon Aug 19 21:37:18 UTC-05:00 2024,Publisher,https://www.wgbh.org/news/local/2024-08-19/as-...


In [59]:
file_path = "./../data_set/Articles_Nov_2020_March_2023.csv"
df2 = pd.read_csv(file_path)
df2 = preprocess_df(df2)
df2.to_csv("./../data_set/combined_data_2.csv", index=False)
df2

df initial size: (12905, 6)
df size after removing empty rows: (12586, 6)
df size after removing rows with invalid dates: (12472, 6)
df final size: (12472, 6)


,Byline,Body,Headline,Publish Date,Publisher,Paths
0,Adam Reilly,"If you’ve been around long enough, you still t...",LISTEN: Election 2020 — The Night(s) Before An...,Sun Nov 01 12:24:58 UTC-05:00 2020,Publisher,/politics/2020/11/01/listen-election-2020-the-...
1,Craig LeMoult,The state Department of Public Health released...,Household 'Clusters' Are A Problem In Massachu...,Sun Nov 01 12:51:24 UTC-05:00 2020,Publisher,/local-news/2020/11/01/household-clusters-are-...
2,Alana Wise,President Trump makes five stops in five diffe...,Trump And Biden Make 11th Hour Election Appeal...,Sun Nov 01 13:17:00 UTC-05:00 2020,Publisher,/politics/2020/11/01/trump-and-biden-make-11th...
3,"Arun Rath, Matt Baskin","<i>For the past year and a half, 20-year-old J...",20-Year-Old Massachusetts Man With Development...,Sun Nov 01 14:27:23 UTC-05:00 2020,Publisher,/local-news/2020/11/01/20-year-old-massachuset...
4,Matthew S. Schwartz,President Trump is celebrating a caravan of su...,Trump Speaks Fondly Of Supporters 'Protecting'...,Sun Nov 01 15:52:00 UTC-05:00 2020,Publisher,/politics/2020/11/02/trump-speaks-fondly-of-su...
...,...,...,...,...,...,...
12467,"Emily Olson, Emma Bowman",Former president Donald Trump has been indicte...,5 key takeaways from the Trump indictment news,Fri Mar 31 06:26:00 UTC-04:00 2023,Publisher,/national-news/2023/03/31/5-key-takeaways-from...
12468,"Dustin Jones, Kaitlyn Radde",Former President Donald Trump was indicted Thu...,Why Trump isn't the first president to face ar...,Fri Mar 31 09:05:00 UTC-04:00 2023,Publisher,/national-news/2023/03/31/why-trump-isnt-the-f...
12469,Ari Daniel,Human attempts to kill cockroaches with sugary...,These cockroaches tweaked their mating rituals...,Fri Mar 31 10:02:00 UTC-04:00 2023,Publisher,/news/2023/03/31/these-cockroaches-tweaked-the...
12470,Alexi Cohan,Artificial intelligence has advanced rapidly i...,Harvard professor says government should pause...,Fri Mar 31 11:06:14 UTC-04:00 2023,Publisher,/science-and-technology/2023/03/31/harvard-pro...


In [81]:
folder_path = "./../data_set/gbh-rss-feed"
df3 = read_folder(folder_path)
df3 = preprocess_df(df3)
df3.to_csv("./../data_set/combined_data_3.csv", index=False)
df3

Publisher column not found. Filling with 'Publisher'
Byline column not found. Filling with 'Author'
df initial size: (2560, 6)
df size after removing empty rows: (2560, 6)
df size after removing rows with invalid dates: (2560, 6)
df final size: (1139, 6)


,Byline,Body,Headline,Publish Date,Publisher,Paths
0,Author,Today on Boston Public Radio:NBC Political Dir...,"Boston Public Radio full show: Feb. 16, 2023",Tue Feb 21 11:11:33 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/local-news/2023/02/2...
1,Author,Today on Boston Public Radio:Boston Mayor Mich...,"Boston Public Radio full show: Feb. 14, 2023",Tue Feb 21 11:12:29 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/local-news/2023/02/2...
2,Author,In the midst of the humanitarian crisis that's...,Lentil soup comes to the rescue in quake-ravag...,Tue Feb 21 13:19:00 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/international-news/2...
3,Author,Consumers are willing to pay monthly subscript...,Arby's+? More restaurants try subscription pro...,Tue Feb 21 14:24:00 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/national-news/2023/0...
4,Author,As military members and elected officials gath...,"With Healey and Driscoll away, it’s acting Gov...",Tue Feb 21 14:42:37 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/politics/2023/02/21/...
...,...,...,...,...,...,...
1134,Author,Gov. Maura Healey signed a $56 billion state b...,Free school meals now permanent in Mass. with ...,Wed Aug 09 16:26:29 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/politics/2023/08/09/...
1135,Author,WASHINGTON — In a sign of growing strains betw...,Biden orders restrictions on U.S. investments ...,Wed Aug 09 17:08:00 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/international-news/2...
1136,Author,Cases of unruly airline passenger behavior hav...,There has been an 80% drop in cases of unruly ...,Wed Aug 09 18:04:00 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/national-news/2023/0...
1137,Author,A Utah man who was accused of making threats t...,Man accused of threatening Biden shot and kill...,Wed Aug 09 18:30:00 UTC-05:00 2023,Publisher,https://www.wgbh.org/news/national-news/2023/0...


In [82]:
combined_df = pd.concat([df1, df2, df3], ignore_index=True)
combined_df = preprocess_df(combined_df)
combined_df.to_csv("./../data_set/all_data.csv", index=False)

Publisher column not found. Filling with 'Publisher'
Byline column not found. Filling with 'Author'
df initial size: (19596, 6)
df size after removing empty rows: (19596, 6)
df size after removing rows with invalid dates: (19596, 6)
df final size: (17247, 6)


In [69]:
def count_articles_by_month(df):
    df['Publish Date'] = pd.to_datetime(df['Publish Date'], errors='coerce').dt.tz_localize(None)
    df2 = df.dropna(subset=['Publish Date']).copy()
    
    df2['Month'] = df2['Publish Date'].dt.to_period('M')
    articles_by_month = df2.groupby('Month').size()
    
    articles_by_month_df = articles_by_month.reset_index(name='Article Count')
    
    return articles_by_month_df

In [70]:
monthly_article_counts = count_articles_by_month(combined_df)
monthly_article_counts

,Month,Article Count
0,2020-11,663
1,2020-12,585
2,2021-01,669
3,2021-02,553
4,2021-03,264
5,2021-11,268
6,2021-12,387
7,2022-01,397
8,2022-02,365
9,2022-03,188


In [85]:
file_path = "./../data_set/all_data.csv"
split_csv_into_batches(file_path)

Publisher column not found. Filling with 'Publisher'
Byline column not found. Filling with 'Author'
df initial size: (17247, 6)
df size after removing empty rows: (17247, 6)
df size after removing rows with invalid dates: (17247, 6)
df final size: (17247, 6)
Saved ./../data_set/batches/batch_1.csv with 1000 articles.
Saved ./../data_set/batches/batch_2.csv with 1000 articles.
Saved ./../data_set/batches/batch_3.csv with 1000 articles.
Saved ./../data_set/batches/batch_4.csv with 1000 articles.
Saved ./../data_set/batches/batch_5.csv with 1000 articles.
Saved ./../data_set/batches/batch_6.csv with 1000 articles.
Saved ./../data_set/batches/batch_7.csv with 1000 articles.
Saved ./../data_set/batches/batch_8.csv with 1000 articles.
Saved ./../data_set/batches/batch_9.csv with 1000 articles.
Saved ./../data_set/batches/batch_10.csv with 1000 articles.
Saved ./../data_set/batches/batch_11.csv with 1000 articles.
Saved ./../data_set/batches/batch_12.csv with 1000 articles.
Saved ./../data_se